# AfriVoices ASR — Fine-tune Whisper-small on all 6 languages (Google Colab)
Training data: Swahili · Somali · Kikuyu · Luo · Maasai · Kalenjin
Processed records are cached to Google Drive so downloads only happen once.

In [ ]:
# ── CELL 1 — Mount Drive + Install ──────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/afrivoices', exist_ok=True)
os.makedirs('/content/drive/MyDrive/afrivoices/records', exist_ok=True)

# Pin datasets to avoid v3.x loading-script breakage
!pip install -q transformers accelerate 'datasets==2.20.0' evaluate jiwer \
               soundfile librosa pydub huggingface_hub kagglehub
print('Dependencies installed.')

In [ ]:

# ── CELL 2 — Secrets + Imports + Drive cache helpers ────────────────────────
from google.colab import userdata
import os, io, glob, json, tarfile, pickle, warnings
import numpy as np
import pandas as pd
import torch
import soundfile as sf
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from transformers import (
    WhisperProcessor, WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments, Seq2SeqTrainer,
)
import evaluate
from huggingface_hub import login, HfApi, hf_hub_download, list_repo_files

HF_TOKEN         = userdata.get('HF_TOKEN')
KAGGLE_API_TOKEN = userdata.get('KAGGLE_API_TOKEN')
if not HF_TOKEN:         raise RuntimeError('Add HF_TOKEN to Colab Secrets (🔑 sidebar)')
if not KAGGLE_API_TOKEN: raise RuntimeError('Add KAGGLE_API_TOKEN to Colab Secrets')

os.environ['KAGGLE_API_TOKEN'] = KAGGLE_API_TOKEN
login(token=HF_TOKEN)
print('HuggingFace login OK.')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

MODEL_ID       = 'openai/whisper-small'
DRIVE_DIR      = '/content/drive/MyDrive/afrivoices'
RECORDS_DIR    = f'{DRIVE_DIR}/records'
CHECKPOINT_DIR = f'{DRIVE_DIR}/whisper-small-all6'
os.makedirs(RECORDS_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'Checkpoint dir : {CHECKPOINT_DIR}')
print(f'Records cache  : {RECORDS_DIR}')

# ── Drive cache helpers ───────────────────────────────────────────────────────
def save_records(records, path):
    """Save records to Drive as float16 (~480 KB/clip).
    Records are already float16 in memory so copy=False avoids a second allocation."""
    packed = [{'input_features': r['input_features'].astype(np.float16, copy=False),
               'labels': r['labels']} for r in records]
    with open(path, 'wb') as f:
        pickle.dump(packed, f, protocol=4)
    print(f'  → {len(records)} records saved ({os.path.getsize(path)/1e6:.0f} MB)')

def load_records(path):
    """Load cached records keeping float16 in RAM.
    11k clips as float16 = ~5.3 GB vs ~10.5 GB as float32.
    The DataCollator converts to float32 per batch on the fly."""
    with open(path, 'rb') as f:
        return pickle.load(f)


In [ ]:
# ── CELL 3 — Load Whisper processor ─────────────────────────────────────────
processor         = WhisperProcessor.from_pretrained(MODEL_ID)
feature_extractor = processor.feature_extractor
tokenizer         = processor.tokenizer
print(f'Processor loaded: {MODEL_ID}')

In [ ]:
# ── CELL 4 — Audio decode helper (single definition used throughout) ─────────
def decode_audio(audio_field):
    """Accept HF Audio dict, bytes dict, or raw bytes → 16 kHz float32 array."""
    if isinstance(audio_field, dict) and 'array' in audio_field:
        arr = np.array(audio_field['array'], dtype=np.float32)
        sr  = audio_field.get('sampling_rate', 16000)
        if sr != 16000:
            import librosa
            arr = librosa.resample(arr, orig_sr=sr, target_sr=16000)
        return arr
    raw = audio_field.get('bytes') if isinstance(audio_field, dict) else audio_field
    if isinstance(raw, bytes) and len(raw) > 0:
        try:
            arr, sr = sf.read(io.BytesIO(raw), dtype='float32')
            if arr.ndim > 1:
                arr = arr.mean(axis=1)
            if sr != 16000:
                import librosa
                arr = librosa.resample(arr, orig_sr=sr, target_sr=16000)
            return arr
        except Exception:
            pass
        from pydub import AudioSegment
        seg = (AudioSegment.from_file(io.BytesIO(raw))
               .set_frame_rate(16000).set_channels(1))
        return np.array(seg.get_array_of_samples(), dtype=np.float32) / 32768.0
    raise ValueError(
        f'Cannot decode audio — type {type(audio_field)}, '
        f'keys: {list(audio_field.keys()) if isinstance(audio_field, dict) else "N/A"}'
    )

print('decode_audio helper ready.')

In [ ]:
# ── CELL 5 — Load Swahili training data (cache-first) ───────────────────────
SWA_CACHE = f'{RECORDS_DIR}/swa_records.pkl'

if os.path.exists(SWA_CACHE):
    print('Loading Swahili from Drive cache...')
    swa_records = load_records(SWA_CACHE)
    print(f'Swahili: {len(swa_records)} clips loaded from cache.')
else:
    N_SAMPLES = 5000
    SHARD     = 0
    print(f'Downloading {N_SAMPLES} Swahili clips from shard {SHARD}...')

    manifest_path = hf_hub_download(
        repo_id='DigitalUmuganda/Afrivoice_Swahili',
        filename=f'agriculture_swahili_train/manifest_{SHARD}.jsonl',
        repo_type='dataset', token=HF_TOKEN,
    )
    with open(manifest_path) as f:
        all_entries = [json.loads(l) for l in f]

    wanted = {}
    for entry in all_entries:
        text = (entry.get('normalized_transcription') or '').strip()
        if text:
            wanted[entry['key']] = text
        if len(wanted) >= N_SAMPLES:
            break
    print(f'  {len(wanted)} valid manifest entries.')

    audio_tar_path = hf_hub_download(
        repo_id='DigitalUmuganda/Afrivoice_Swahili',
        filename=f'agriculture_swahili_train/audio/audio_{SHARD}.tar.xz',
        repo_type='dataset', token=HF_TOKEN,
    )
    print('  Audio tar downloaded. Processing...')

    swa_records = []
    tokenizer.set_prefix_tokens(language='swahili', task='transcribe')

    with tarfile.open(audio_tar_path, 'r:xz') as tar:
        for member in tar:
            if not member.name.endswith('.webm'):
                continue
            key = os.path.basename(member.name).replace('.webm', '')
            if key not in wanted:
                continue
            try:
                arr = decode_audio({'bytes': tar.extractfile(member).read()})
                arr = arr[:480_000]
            except Exception:
                continue
            # Store as float16 immediately — keeps RAM at ~480 KB/clip instead of 960 KB
            input_features = feature_extractor(arr, sampling_rate=16000).input_features[0].astype(np.float16)
            labels = tokenizer(wanted[key]).input_ids
            swa_records.append({'input_features': input_features, 'labels': labels})
            if len(swa_records) % 500 == 0:
                print(f'  {len(swa_records)} / {N_SAMPLES} clips', flush=True)
            if len(swa_records) >= N_SAMPLES:
                break

    print(f'\nSwahili: {len(swa_records)} clips ready. Saving to Drive...')
    save_records(swa_records, SWA_CACHE)


In [ ]:
# ── CELL 6 — Load Somali training data (cache-first, multi-shard) ────────────
import lzma
SOM_CACHE  = f'{RECORDS_DIR}/som_records.pkl'
SOM_TARGET = 2000

if os.path.exists(SOM_CACHE):
    print('Loading Somali from Drive cache...')
    som_records = load_records(SOM_CACHE)
    print(f'Somali: {len(som_records)} clips loaded from cache.')
else:
    print('Scanning Somali manifests for valid transcriptions...')
    wanted_by_shard = {}
    total_wanted    = 0

    for shard in range(171):
        manifest_path = hf_hub_download(
            repo_id='DigitalUmuganda/Afrivoice',
            filename=f'Somali/manifest_{shard}.json',
            repo_type='dataset', token=HF_TOKEN,
        )
        with open(manifest_path) as f:
            entries = [json.loads(line) for line in f if line.strip()]

        shard_wanted = {}
        for entry in entries:
            text = (entry.get('transcription') or
                    entry.get('normalized_transcription') or '').strip()
            key  = os.path.splitext(os.path.basename(
                str(entry.get('audio_filepath', ''))))[0]
            if text and key:
                shard_wanted[key] = text

        if shard_wanted:
            wanted_by_shard[shard]  = shard_wanted
            total_wanted           += len(shard_wanted)
            print(f'  shard {shard}: {len(shard_wanted)} valid  (total: {total_wanted})',
                  flush=True)
        if total_wanted >= SOM_TARGET:
            break

    print(f'\nScanned {len(wanted_by_shard)} shards, {total_wanted} valid entries.')

    som_records = []
    tokenizer.set_prefix_tokens(language='somali', task='transcribe')

    for shard, wanted in wanted_by_shard.items():
        if len(som_records) >= SOM_TARGET:
            break
        print(f'\nDownloading audio_{shard}.tar.xz ({len(wanted)} targets)...')
        for attempt in range(2):
            audio_tar_path = hf_hub_download(
                repo_id='DigitalUmuganda/Afrivoice',
                filename=f'Somali/audio_shards/audio_{shard}.tar.xz',
                repo_type='dataset', token=HF_TOKEN,
                force_download=(attempt > 0),
            )
            try:
                with tarfile.open(audio_tar_path, 'r:xz') as tar:
                    for member in tar:
                        if member.isdir():
                            continue
                        key = os.path.splitext(os.path.basename(member.name))[0]
                        if key not in wanted:
                            continue
                        try:
                            arr = decode_audio({'bytes': tar.extractfile(member).read()})
                            arr = arr[:480_000]
                        except Exception:
                            continue
                        # Store as float16 immediately — keeps RAM at ~480 KB/clip instead of 960 KB
                        input_features = feature_extractor(
                            arr, sampling_rate=16000).input_features[0].astype(np.float16)
                        labels = tokenizer(wanted[key]).input_ids
                        som_records.append({'input_features': input_features,
                                            'labels': labels})
                        if len(som_records) % 200 == 0:
                            print(f'  {len(som_records)} Somali clips', flush=True)
                        if len(som_records) >= SOM_TARGET:
                            break
                break
            except (lzma.LZMAError, tarfile.TarError) as e:
                print(f'  Shard {shard} corrupt (attempt {attempt+1}): {e}')
                if attempt == 1:
                    print(f'  Skipping shard {shard}.')

    print(f'\nSomali: {len(som_records)} clips ready. Saving to Drive...')
    save_records(som_records, SOM_CACHE)


In [ ]:
# ── CELL 6b — Load Kikuyu, Luo, Maasai, Kalenjin from MCAA1-MSU/anv_data_ke ─
ANV_CACHE    = f'{RECORDS_DIR}/anv_records.pkl'
TARGET_LANGS = ['kik', 'luo', 'mas', 'kln']
ANV_TARGET   = 1000   # clips per language

if os.path.exists(ANV_CACHE):
    print('Loading ANV data from Drive cache...')
    anv_records = load_records(ANV_CACHE)
    print(f'ANV: {len(anv_records)} clips loaded from cache.')
else:
    print('Listing MCAA1-MSU/anv_data_ke files...')
    all_anv_files  = list(list_repo_files('MCAA1-MSU/anv_data_ke',
                                           repo_type='dataset', token=HF_TOKEN))
    parquet_files  = sorted([f for f in all_anv_files if f.endswith('.parquet')])

    train_files = {lang: [] for lang in TARGET_LANGS}
    for f in parquet_files:
        parts = f.split('/')
        if parts[0] in TARGET_LANGS and parts[1] == 'train':
            train_files[parts[0]].append(f)
    for lang, files in train_files.items():
        print(f'  {lang}: {len(files)} train shards')

    buckets = {lang: [] for lang in TARGET_LANGS}
    # Kikuyu/Luo/Maasai/Kalenjin are not in Whisper's 99-language vocab,
    # so we train without a language prefix token.
    tokenizer.set_prefix_tokens(language=None, task='transcribe')

    for lang in TARGET_LANGS:
        print(f'\nLoading {lang}...')
        for shard_path in train_files[lang]:
            if len(buckets[lang]) >= ANV_TARGET:
                break
            pq_path = hf_hub_download(
                repo_id='MCAA1-MSU/anv_data_ke',
                filename=shard_path,
                repo_type='dataset', token=HF_TOKEN,
            )
            df = pd.read_parquet(pq_path)
            tcol = ('transcription'  if 'transcription'  in df.columns else
                    'actualSentence' if 'actualSentence' in df.columns else
                    'transcript'     if 'transcript'     in df.columns else None)
            if tcol is None:
                continue
            for _, row in df.iterrows():
                if len(buckets[lang]) >= ANV_TARGET:
                    break
                text = (row.get(tcol) or '').strip()
                if not text:
                    continue
                try:
                    arr = decode_audio(row['audio'])
                    arr = arr[:480_000]
                except Exception:
                    continue
                # Store as float16 immediately — keeps RAM at ~480 KB/clip instead of 960 KB
                input_features = feature_extractor(
                    arr, sampling_rate=16000).input_features[0].astype(np.float16)
                labels = tokenizer(text).input_ids
                buckets[lang].append({'input_features': input_features,
                                      'labels': labels})
            print(f'  {shard_path.split("/")[-1]}: {len(buckets[lang])} / {ANV_TARGET}',
                  flush=True)

    anv_records = []
    for lang, recs in buckets.items():
        print(f'  {lang}: {len(recs)} clips')
        anv_records.extend(recs)

    print(f'\nANV total: {len(anv_records)} clips ready. Saving to Drive...')
    save_records(anv_records, ANV_CACHE)


In [ ]:
# ── CELL 7 — Build combined dataset (all 6 languages) + train/eval split ─────
from torch.utils.data import Dataset as TorchDataset

class WhisperDataset(TorchDataset):
    def __init__(self, records):
        self.records = records
    def __len__(self):
        return len(self.records)
    def __getitem__(self, i):
        return self.records[i]

all_records = swa_records + som_records + anv_records
np.random.seed(42)
np.random.shuffle(all_records)

split    = int(0.95 * len(all_records))
train_ds = WhisperDataset(all_records[:split])
eval_ds  = WhisperDataset(all_records[split:])

print(f'Total  : {len(all_records)} clips')
print(f'  swa  : {len(swa_records)}')
print(f'  som  : {len(som_records)}')
print(f'  anv  : {len(anv_records)} (kik+luo+mas+kln)')
print(f'Train  : {len(train_ds)}')
print(f'Eval   : {len(eval_ds)}')

In [ ]:
# ── CELL 8 — Data collator + WER metric ──────────────────────────────────────
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        input_features = torch.tensor(
            np.stack([f['input_features'] for f in features]), dtype=torch.float32
        )
        label_features = [{'input_ids': f['labels']} for f in features]
        labels_batch   = self.processor.tokenizer.pad(
            label_features, return_tensors='pt'
        )
        labels = labels_batch['input_ids'].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]
        return {'input_features': input_features, 'labels': labels}


wer_metric = evaluate.load('wer')

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    pred_str  = tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return {'wer': round(wer_metric.compute(
        predictions=pred_str, references=label_str), 4)}


data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=processor.tokenizer.convert_tokens_to_ids(
        '<|startoftranscript|>'),
)
print('Data collator + WER metric ready.')

In [ ]:
# ── CELL 9 — Load model + training args ──────────────────────────────────────
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
model.config.forced_decoder_ids            = None
model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens    = []   # must be on generation_config, not model.config
print(f'Model loaded: {sum(p.numel() for p in model.parameters())/1e6:.0f}M parameters')

training_args = Seq2SeqTrainingArguments(
    output_dir                  = CHECKPOINT_DIR,
    per_device_train_batch_size = 8,
    gradient_accumulation_steps = 4,    # effective batch = 32
    learning_rate               = 1e-5,
    warmup_steps                = 100,
    max_steps                   = 600,
    gradient_checkpointing      = True,
    fp16                        = True,
    # Adafactor uses ~0.24 GB vs AdamW's ~1.84 GB (no momentum tensors).
    # Critical: with 5.3 GB of records in RAM, AdamW pushes us over 12.7 GB.
    optim                       = 'adafactor',
    eval_strategy               = 'steps',
    per_device_eval_batch_size  = 4,
    predict_with_generate       = True,
    generation_max_length       = 225,
    save_steps                  = 200,
    eval_steps                  = 200,
    save_total_limit            = 3,
    logging_steps               = 25,
    load_best_model_at_end      = True,
    metric_for_best_model       = 'wer',
    greater_is_better           = False,
    report_to                   = ['tensorboard'],
    push_to_hub                 = False,
    dataloader_num_workers      = 0,    # 0 avoids multiprocessing overhead with in-memory data
)
print('Training args configured.')


In [ ]:
# ── CELL 10 — Train ──────────────────────────────────────────────────────────
trainer = Seq2SeqTrainer(
    model            = model,
    args             = training_args,
    train_dataset    = train_ds,
    eval_dataset     = eval_ds,
    data_collator    = data_collator,
    compute_metrics  = compute_metrics,
    processing_class = feature_extractor,
)

print('Starting fine-tuning...')
print(f'  {len(train_ds)} train clips  |  {len(eval_ds)} eval clips')
print(f'  max_steps={training_args.max_steps}, effective batch=32')
trainer.train()
print('Training done.')

# Save to Google Drive
trainer.save_model(CHECKPOINT_DIR)
processor.save_pretrained(CHECKPOINT_DIR)
print(f'Model saved to Drive: {CHECKPOINT_DIR}')

# Push to HuggingFace Hub as backup (private)
api     = HfApi(token=HF_TOKEN)
user    = api.whoami()
REPO_ID = f"{user['name']}/afrivoices-whisper-small-all6"
api.create_repo(REPO_ID, exist_ok=True, private=True)
model.push_to_hub(REPO_ID, token=HF_TOKEN)
processor.push_to_hub(REPO_ID, token=HF_TOKEN)
print(f'Model pushed to HF Hub: https://huggingface.co/{REPO_ID}')

In [ ]:
# ── CELL 11 — Reload fine-tuned model for inference ──────────────────────────
import gc

try:    del trainer
except NameError: pass
try:    del model
except NameError: pass
gc.collect()
torch.cuda.empty_cache()
print(f'GPU memory freed. Reserved: {torch.cuda.memory_reserved()/1e9:.1f} GB')

ft_processor = WhisperProcessor.from_pretrained(CHECKPOINT_DIR)
ft_model     = WhisperForConditionalGeneration.from_pretrained(
    CHECKPOINT_DIR, torch_dtype=torch.float16
).to(device)
ft_model.eval()
ft_model.config.forced_decoder_ids            = None
ft_model.generation_config.forced_decoder_ids = None
ft_model.generation_config.suppress_tokens    = []
# Whisper's generation_config sets max_length=448 by default, which conflicts
# with our per-call max_new_tokens and causes a noisy warning. Clear it here.
ft_model.generation_config.max_length         = None
print(f'Fine-tuned model loaded from {CHECKPOINT_DIR}')


In [ ]:
# ── CELL 12 — Transcribe test set ────────────────────────────────────────────
import kagglehub
import time
from concurrent.futures import ThreadPoolExecutor

def transcribe_batch(arrays, language=None):
    arrays = [a[:480_000] for a in arrays]
    inputs = ft_processor(
        arrays, sampling_rate=16000, return_tensors='pt'
    ).input_features.to(device).to(torch.float16)
    gen_kwargs = {'max_new_tokens': 64}  # 64 tokens ≈ 30 words — enough for any utterance
    if language:
        lang_id = ft_processor.tokenizer.convert_tokens_to_ids(f'<|{language}|>')
        task_id = ft_processor.tokenizer.convert_tokens_to_ids('<|transcribe|>')
        nots_id = ft_processor.tokenizer.convert_tokens_to_ids('<|notimestamps|>')
        gen_kwargs['forced_decoder_ids'] = [[1, lang_id], [2, task_id], [3, nots_id]]
    with torch.no_grad():
        ids = ft_model.generate(input_features=inputs, **gen_kwargs)
    return ft_processor.batch_decode(ids, skip_special_tokens=True)


def safe_decode(row):
    try:
        return row.id, row.language, decode_audio(row.audio)
    except Exception as e:
        return row.id, row.language, None


LANG_TO_WHISPER = {
    'swa': 'sw',
    'som': 'so',
    'kik': None,
    'luo': None,
    'mas': None,
    'kln': None,
}

test_path       = kagglehub.dataset_download('digitalumuganda/anv-test-data-nt')
CHECKPOINT_FILE = f'{DRIVE_DIR}/submission_checkpoint.csv'
BATCH_SIZE      = 32
SAVE_EVERY      = 5
print(f'Test data at: {test_path}')

if os.path.exists(CHECKPOINT_FILE):
    existing  = pd.read_csv(CHECKPOINT_FILE)
    empty_pct = (existing['transcription'].isna() |
                 (existing['transcription'].str.strip() == '')).mean()
    if empty_pct > 0.5:
        os.remove(CHECKPOINT_FILE)
        print(f'Deleted corrupted checkpoint ({empty_pct:.0%} empty). Starting fresh.')
        results, done_ids = [], set()
    else:
        results  = existing.to_dict('records')
        done_ids = set(existing['id'])
        print(f'Resuming: {len(results)} clips done.')
else:
    results, done_ids = [], set()
    print('Starting fresh.')

all_parquet_files = sorted(
    glob.glob(os.path.join(test_path, '**', '*.parquet'), recursive=True)
)
print(f'{len(all_parquet_files)} parquet files. Batch size: {BATCH_SIZE}.\n')

t0 = time.time()
for pf_idx, pq_file in enumerate(all_parquet_files):
    df = pd.read_parquet(pq_file)
    df = df[~df['id'].isin(done_ids)]
    if len(df) == 0:
        print(f'[{pf_idx+1}/{len(all_parquet_files)}] already done — skip')
        continue

    lang3   = df['language'].iloc[0]
    wh_lang = LANG_TO_WHISPER.get(lang3)
    elapsed = (time.time() - t0) / 60
    done_files = pf_idx - sum(
        1 for p in all_parquet_files[:pf_idx]
        if pd.read_parquet(p)['id'].isin(done_ids).all()
    ) if pf_idx > 0 else 0
    print(f'[{pf_idx+1}/{len(all_parquet_files)}] {os.path.basename(pq_file)} '
          f'lang={lang3} rows={len(df)}  ({elapsed:.1f} min elapsed)', flush=True)

    rows = list(df.itertuples(index=False))
    for i in range(0, len(rows), BATCH_SIZE):
        chunk = rows[i: i + BATCH_SIZE]
        # Decode audio in parallel with threads (I/O-bound: soundfile reads bytes)
        with ThreadPoolExecutor(max_workers=4) as ex:
            decoded = list(ex.map(safe_decode, chunk))

        arrays, batch_ids, batch_langs = [], [], []
        for id_, lang_, arr in decoded:
            if arr is None:
                results.append({'id': id_, 'language': lang_, 'transcription': '.'})
                done_ids.add(id_)
            else:
                arrays.append(arr)
                batch_ids.append(id_)
                batch_langs.append(lang_)

        if arrays:
            try:
                texts = transcribe_batch(arrays, language=wh_lang)
                for id_, lang_, text in zip(batch_ids, batch_langs, texts):
                    results.append({'id': id_, 'language': lang_,
                                    'transcription': text.strip() or '.'})
                    done_ids.add(id_)
            except Exception as e:
                print(f'  BATCH ERROR ({e}) — falling back to one-by-one')
                for id_, lang_, arr in zip(batch_ids, batch_langs, arrays):
                    try:
                        text = transcribe_batch([arr], language=wh_lang)[0].strip() or '.'
                    except Exception as e2:
                        print(f'    FAILED {id_}: {e2}')
                        text = '.'
                    results.append({'id': id_, 'language': lang_,
                                    'transcription': text})
                    done_ids.add(id_)

    if (pf_idx + 1) % SAVE_EVERY == 0 or (pf_idx + 1) == len(all_parquet_files):
        pd.DataFrame(results).to_csv(CHECKPOINT_FILE, index=False)
        eta = ((time.time() - t0) / (pf_idx + 1)) * (len(all_parquet_files) - pf_idx - 1) / 60
        print(f'  → checkpoint saved ({len(results)} total) | ETA: {eta:.0f} min', flush=True)

print(f'\nDone. Total: {len(results)} in {(time.time()-t0)/60:.1f} min')


In [ ]:
# ── CELL 13 — Build submission file ──────────────────────────────────────────
SUBMISSION_PATH = f'{DRIVE_DIR}/submission.csv'

if not results and os.path.exists(CHECKPOINT_FILE):
    submission_df = pd.read_csv(CHECKPOINT_FILE)
else:
    submission_df = pd.DataFrame(results)

mask = (submission_df['transcription'].isna() |
        (submission_df['transcription'].str.strip() == ''))
if mask.sum() > 0:
    print(f'Replacing {mask.sum()} empty rows with \'.\'')
    submission_df.loc[mask, 'transcription'] = '.'

submission_df = submission_df[['id', 'language', 'transcription']]
submission_df.to_csv(SUBMISSION_PATH, index=False)

sub = pd.read_csv(SUBMISSION_PATH)
print(f'Rows             : {len(sub)}')
print(f'NaN transcription: {sub["transcription"].isna().sum()}')
print(f'Empty            : {(sub["transcription"].str.strip() == "").sum()}')
print('\nRows per language:')
print(sub['language'].value_counts().to_string())
print(sub.head())
print(f'\nsubmission.csv saved to Drive: {SUBMISSION_PATH}')
print('Download it and upload to the competition: Submit Prediction')